# Modelos y Grid Search

Este notebook entrena modelos de detección de fraude a partir del análisis exploratorio (`01_EDA`) y del dataset limpio generado en `03_Preprocesamiento`.

El objetivo principal es comparar modelos base y después ajustar hiperparámetros con `GridSearchCV`, usando `average_precision` como métrica principal porque la clase fraude está muy desbalanceada.

## Librerías y configuración

In [55]:
from pathlib import Path
import time
import warnings

import joblib
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    fbeta_score,
    make_scorer,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
CV_FOLDS = 3
N_JOBS = -1
RUN_SMOTE_GRID = True

## Carga de datos limpios

Se utilizan `X_clean.csv` e `y.csv`, generados en el notebook de preprocesamiento después de limpiar `age` y `gender`, crear variables como `high_amount`, `day_of_week_sin`, `day_of_week_cos` y `merchant_freq`, y eliminar columnas sin variabilidad o ya transformadas.

In [56]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_path = project_root / "data" / "clean"
models_path = project_root / "models"
models_path.mkdir(parents=True, exist_ok=True)

X = pd.read_csv(data_path / "X_clean.csv")
y = pd.read_csv(data_path / "y.csv").squeeze("columns")

print("X:", X.shape)
print("y:", y.shape)
display(y.value_counts().rename("n"))
display((y.value_counts(normalize=True) * 100).round(3).rename("%"))
X.head()

X: (594643, 10)
y: (594643,)


fraud
0    587443
1      7200
Name: n, dtype: int64

fraud
0    98.789
1     1.211
Name: %, dtype: float64

,customer,age,gender,merchant,category,amount,high_amount,day_of_week_sin,day_of_week_cos,merchant_freq
0,'C1093826151',4,M,'M348934600','es_transportation',4.55,0,0.0,1.0,205426
1,'C352968107',2,M,'M348934600','es_transportation',39.68,0,0.0,1.0,205426
2,'C2054744914',4,F,'M1823072687','es_transportation',26.89,0,0.0,1.0,299693
3,'C1760612790',3,M,'M348934600','es_transportation',17.25,0,0.0,1.0,205426
4,'C757503768',5,M,'M348934600','es_transportation',35.72,0,0.0,1.0,205426


## División train/test

Se usa una división estratificada para conservar la proporción de fraude en entrenamiento y test.

In [57]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Fraude en train:", round(y_train.mean() * 100, 3), "%")
print("Fraude en test :", round(y_test.mean() * 100, 3), "%")

Train: (475714, 10) Test: (118929, 10)
Fraude en train: 1.211 %
Fraude en test : 1.211 %


## Preprocesamiento para modelado

Las variables categóricas se codifican con `OneHotEncoder`. Para `customer` se agrupan categorías poco frecuentes, reduciendo dimensionalidad y mejorando la generalización ante clientes raros. Las numéricas se escalan con `StandardScaler`.

In [58]:
cat_cols = ["customer", "age", "gender", "merchant", "category"]
num_cols = [col for col in X.columns if col not in cat_cols]

cat_idx = [X.columns.get_loc(col) for col in cat_cols]
num_idx = [X.columns.get_loc(col) for col in num_cols]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=20,
                sparse_output=True,
            ),
            cat_idx,
        ),
        ("num", StandardScaler(), num_idx),
    ],
    remainder="drop",
)

print("Variables categóricas:", cat_cols)
print("Índices categóricos:", cat_idx)
print("Variables numéricas:", num_cols)
print("Índices numéricos:", num_idx)

Variables categóricas: ['customer', 'age', 'gender', 'merchant', 'category']
Índices categóricos: [0, 1, 2, 3, 4]
Variables numéricas: ['amount', 'high_amount', 'day_of_week_sin', 'day_of_week_cos', 'merchant_freq']
Índices numéricos: [5, 6, 7, 8, 9]


## Funciones de evaluación

In [59]:
def format_seconds(seconds):
    minutes, sec = divmod(float(seconds), 60)
    hours, minutes = divmod(int(minutes), 60)
    if hours:
        return f"{hours}h {minutes:02d}m {sec:04.1f}s"
    if minutes:
        return f"{minutes}m {sec:04.1f}s"
    return f"{sec:.1f}s"


def evaluate_model(name, estimator, X_eval, y_eval):
    y_proba = estimator.predict_proba(X_eval)[:, 1]
    y_pred = estimator.predict(X_eval)

    return {
        "model": name,
        "roc_auc": roc_auc_score(y_eval, y_proba),
        "pr_auc": average_precision_score(y_eval, y_proba),
        "f1": fbeta_score(y_eval, y_pred, beta=1),
        "f2": fbeta_score(y_eval, y_pred, beta=2),
        "confusion_matrix": confusion_matrix(y_eval, y_pred),
        "classification_report": classification_report(
            y_eval,
            y_pred,
            target_names=["legitima", "fraude"],
        ),
    }


def summarize_results(results):
    table = pd.DataFrame(results).T

    if "fit_time_sec" in table.columns:
        table["fit_time_sec"] = table["fit_time_sec"].astype(float)
        table["fit_time_min"] = table["fit_time_sec"] / 60
        table["fit_time"] = table["fit_time_sec"].map(format_seconds)

    metric_cols = [
        "roc_auc",
        "pr_auc",
        "best_cv_pr_auc",
        "f1",
        "f2",
        "fit_time_sec",
        "fit_time_min",
    ]
    existing_metric_cols = [col for col in metric_cols if col in table.columns]
    table[existing_metric_cols] = table[existing_metric_cols].astype(float).round(4)
    return table.sort_values("pr_auc", ascending=False)

## Modelos base

Primero se entrenan configuraciones razonables sin búsqueda, para tener una referencia rápida antes del Grid Search.

In [60]:
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos
print(f"Ratio neg/pos: {scale_pos_weight:.2f}")

base_models = {
    "RandomForest_base": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    ),
    "LightGBM_base": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        verbose=-1,
    ),
    "XGBoost_base": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="aucpr",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    ),
}

Ratio neg/pos: 81.59


In [61]:
base_results = {}
base_estimators = {}

for name, model in base_models.items():
    pipe = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", model),
        ]
    )

    start = time.perf_counter()
    pipe.fit(X_train, y_train)
    fit_time = time.perf_counter() - start

    metrics = evaluate_model(name, pipe, X_test, y_test)
    metrics["fit_time_sec"] = fit_time

    base_results[name] = metrics
    base_estimators[name] = pipe

    print(
        f"{name}: PR-AUC={metrics['pr_auc']:.4f} | "
        f"ROC-AUC={metrics['roc_auc']:.4f} | F2={metrics['f2']:.4f} | "
        f"train={fit_time:.1f}s"
    )

base_results_df = summarize_results(base_results)
display(base_results_df[["model", "roc_auc", "pr_auc", "f1", "f2", "fit_time_sec"]])

RandomForest_base: PR-AUC=0.8499 | ROC-AUC=0.9972 | F2=0.6652 | train=6.0s
LightGBM_base: PR-AUC=0.9161 | ROC-AUC=0.9982 | F2=0.7772 | train=3.8s
XGBoost_base: PR-AUC=0.9010 | ROC-AUC=0.9979 | F2=0.7031 | train=1.8s


,model,roc_auc,pr_auc,f1,f2,fit_time_sec
LightGBM_base,LightGBM_base,0.9982,0.9161,0.5982,0.7772,3.8144
XGBoost_base,XGBoost_base,0.9979,0.9010,0.4936,0.7031,1.8490
RandomForest_base,RandomForest_base,0.9972,0.8499,0.4475,0.6652,5.9877


## Grid Search con y sin SMOTE

Se ajustan Random Forest, LightGBM y XGBoost con grids más amplios que incluyen más hiperparámetros relevantes. Cada modelo se prueba en dos escenarios: sin SMOTE y con SMOTE.

La métrica principal sigue siendo `average_precision` / PR-AUC, más adecuada que ROC-AUC cuando la clase fraude es minoritaria.

Nota sobre SMOTE: para este dataset se usa `SMOTENC`, una variante de SMOTE preparada para combinar variables categóricas y numéricas. Se aplica antes del one-hot encoding para no interpolar directamente columnas categóricas codificadas. Aun así, puede aumentar tiempo y memoria, por eso la tabla final incluye tanto métricas como tiempo de entrenamiento.

In [62]:
scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "f2": make_scorer(fbeta_score, beta=2),
}

cv = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

smote_param_grid = {
    "smote__sampling_strategy": [0.05, 0.10],
    "smote__k_neighbors": [3, 5],
}

grid_configs = {
    "RandomForest": {
        "estimator": RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
        "param_grid": {
            "classifier__n_estimators": [200, 400],
            "classifier__max_depth": [15, 25, None],
            "classifier__min_samples_split": [2, 10],
            "classifier__min_samples_leaf": [1, 5],
            "classifier__max_features": ["sqrt", 0.5],
            "classifier__class_weight": ["balanced_subsample"],
        },
    },
    "LightGBM": {
        "estimator": LGBMClassifier(
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbose=-1,
        ),
        "param_grid": {
            "classifier__n_estimators": [300, 600],
            "classifier__learning_rate": [0.03, 0.07],
            "classifier__num_leaves": [31, 63],
            "classifier__max_depth": [-1, 8],
            "classifier__min_child_samples": [20, 50],
            "classifier__subsample": [0.8],
            "classifier__colsample_bytree": [0.8],
            "classifier__class_weight": ["balanced", None],
        },
    },
    "XGBoost": {
        "estimator": XGBClassifier(
            random_state=RANDOM_STATE,
            n_jobs=1,
            eval_metric="aucpr",
            tree_method="hist",
        ),
        "param_grid": {
            "classifier__n_estimators": [300, 600],
            "classifier__learning_rate": [0.03, 0.07],
            "classifier__max_depth": [3, 5],
            "classifier__min_child_weight": [1, 5],
            "classifier__subsample": [0.8, 1.0],
            "classifier__colsample_bytree": [0.8],
            "classifier__scale_pos_weight": [scale_pos_weight],
        },
    },
}

for model_name, config in grid_configs.items():
    base_combinations = int(np.prod([len(values) for values in config["param_grid"].values()]))
    smote_combinations = base_combinations * int(np.prod([len(values) for values in smote_param_grid.values()]))
    print(f"{model_name}: {base_combinations} combinaciones sin SMOTE")
    if RUN_SMOTE_GRID:
        print(f"{model_name}: {smote_combinations} combinaciones con SMOTE")

RandomForest: 48 combinaciones sin SMOTE
RandomForest: 192 combinaciones con SMOTE
LightGBM: 64 combinaciones sin SMOTE
LightGBM: 256 combinaciones con SMOTE
XGBoost: 32 combinaciones sin SMOTE
XGBoost: 128 combinaciones con SMOTE


In [63]:
def build_pipeline(model, use_smote):
    if use_smote:
        return ImbPipeline(
            steps=[
                ("smote", SMOTENC(categorical_features=cat_idx, random_state=RANDOM_STATE)),
                ("preprocessor", preprocessor),
                ("classifier", model),
            ]
        )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", model),
        ]
    )


def run_grid_searches(grid_configs, X_train, y_train, X_test, y_test):
    grid_results = {}
    best_estimators = {}
    cv_results = {}

    for model_name, config in grid_configs.items():
        smote_options = [False, True] if RUN_SMOTE_GRID else [False]

        for use_smote in smote_options:
            tag = f"{model_name}_{'con_smote' if use_smote else 'sin_smote'}"
            pipe = build_pipeline(config["estimator"], use_smote)

            param_grid = dict(config["param_grid"])
            if use_smote:
                param_grid.update(smote_param_grid)

            grid = GridSearchCV(
                estimator=pipe,
                param_grid=param_grid,
                scoring=scoring,
                refit="pr_auc",
                cv=cv,
                n_jobs=N_JOBS,
                verbose=2,
                return_train_score=False,
                error_score=np.nan,
            )

            print("=" * 80)
            print(f"Grid Search: {tag}")
            print("=" * 80)

            start = time.perf_counter()
            grid.fit(X_train, y_train)
            fit_time = time.perf_counter() - start

            metrics = evaluate_model(tag, grid.best_estimator_, X_test, y_test)
            metrics["algorithm"] = model_name
            metrics["smote"] = use_smote
            metrics["fit_time_sec"] = fit_time
            metrics["best_cv_pr_auc"] = grid.best_score_
            metrics["best_params"] = grid.best_params_

            grid_results[tag] = metrics
            best_estimators[tag] = grid.best_estimator_
            cv_results[tag] = pd.DataFrame(grid.cv_results_).sort_values(
                "rank_test_pr_auc"
            )

            print(f"Mejor CV PR-AUC: {grid.best_score_:.4f}")
            print(f"Test PR-AUC: {metrics['pr_auc']:.4f}")
            print(f"Test ROC-AUC: {metrics['roc_auc']:.4f}")
            print(f"Test F2: {metrics['f2']:.4f}")
            print(f"Tiempo: {format_seconds(fit_time)}")
            print("Mejores parámetros:", grid.best_params_)

    return grid_results, best_estimators, cv_results

In [64]:
grid_results, grid_estimators, cv_results = run_grid_searches(
    grid_configs,
    X_train,
    y_train,
    X_test,
    y_test,
)

grid_results_df = summarize_results(grid_results)
display(
    grid_results_df[
        [
            "algorithm",
            "smote",
            "roc_auc",
            "pr_auc",
            "best_cv_pr_auc",
            "f1",
            "f2",
            "fit_time",
            "fit_time_min",
        ]
    ]
)

Grid Search: RandomForest_sin_smote
Fitting 3 folds for each of 48 candidates, totalling 144 fits
[CV] END classifier__class_weight=balanced_subsample, classifier__max_depth=15, classifier__max_features=sqrt, classifier__min_samples_leaf=1, classifier__min_samples_split=10, classifier__n_estimators=200; total time=  14.4s
[CV] END classifier__class_weight=balanced_subsample, classifier__max_depth=15, classifier__max_features=sqrt, classifier__min_samples_leaf=1, classifier__min_samples_split=10, classifier__n_estimators=200; total time=  18.7s
[CV] END classifier__class_weight=balanced_subsample, classifier__max_depth=15, classifier__max_features=sqrt, classifier__min_samples_leaf=1, classifier__min_samples_split=10, classifier__n_estimators=200; total time=  19.0s
[CV] END classifier__class_weight=balanced_subsample, classifier__max_depth=15, classifier__max_features=sqrt, classifier__min_samples_leaf=1, classifier__min_samples_split=2, classifier__n_estimators=200; total time=  19.5s

,algorithm,smote,roc_auc,pr_auc,best_cv_pr_auc,f1,f2,fit_time,fit_time_min
LightGBM_con_smote,LightGBM,True,0.9984,0.9269,0.9275,0.8542,0.8433,41m 34.0s,41.5667
LightGBM_sin_smote,LightGBM,False,0.9985,0.9266,0.9261,0.8510,0.8240,6m 36.4s,6.6068
XGBoost_sin_smote,XGBoost,False,0.9981,0.9121,0.9156,0.5719,0.7601,59.2s,0.9861
XGBoost_con_smote,XGBoost,True,0.9979,0.9109,0.9140,0.5121,0.7161,14m 13.4s,14.2231
RandomForest_sin_smote,RandomForest,False,0.9972,0.8966,0.8993,0.6788,0.8100,22m 06.4s,22.1072
RandomForest_con_smote,RandomForest,True,0.9974,0.8931,0.8928,0.6725,0.8067,1h 57m 54.3s,117.9051


## Comparación final y guardado del mejor modelo

In [65]:
all_results = {**base_results, **grid_results}
all_estimators = {**base_estimators, **grid_estimators}

comparison_df = summarize_results(all_results)
comparison_cols = [
    "model",
    "algorithm",
    "smote",
    "roc_auc",
    "pr_auc",
    "best_cv_pr_auc",
    "f1",
    "f2",
    "fit_time",
    "fit_time_min",
    "fit_time_sec",
]
comparison_cols = [col for col in comparison_cols if col in comparison_df.columns]
display(comparison_df[comparison_cols])

if "smote" in comparison_df.columns:
    smote_summary = (
        comparison_df.dropna(subset=["smote"])
        .groupby(["algorithm", "smote"])[["pr_auc", "f2", "fit_time_min"]]
        .max()
        .round(4)
    )
    print("Comparativa resumida con/sin SMOTE:")
    display(smote_summary)

best_name = comparison_df.index[0]
best_pipeline = all_estimators[best_name]

print("Mejor modelo:", best_name)
print("PR-AUC test:", round(comparison_df.loc[best_name, "pr_auc"], 4))
print("ROC-AUC test:", round(comparison_df.loc[best_name, "roc_auc"], 4))
print("F2 test:", round(comparison_df.loc[best_name, "f2"], 4))
print("Tiempo entrenamiento:", comparison_df.loc[best_name, "fit_time"])
print("\nClassification report:")
print(all_results[best_name]["classification_report"])
print("Matriz de confusión:")
print(all_results[best_name]["confusion_matrix"])

comparison_output = comparison_df.drop(columns=["confusion_matrix", "classification_report"], errors="ignore")
comparison_output.to_csv(models_path / "model_comparison.csv", index=True)
joblib.dump(best_pipeline, models_path / "best_fraud_pipeline.joblib")

print("Resultados guardados en:", models_path / "model_comparison.csv")
print("Modelo guardado en:", models_path / "best_fraud_pipeline.joblib")

,model,algorithm,smote,roc_auc,pr_auc,best_cv_pr_auc,f1,f2,fit_time,fit_time_min,fit_time_sec
LightGBM_con_smote,LightGBM_con_smote,LightGBM,True,0.9984,0.9269,0.9275,0.8542,0.8433,41m 34.0s,41.5667,2494.0021
LightGBM_sin_smote,LightGBM_sin_smote,LightGBM,False,0.9985,0.9266,0.9261,0.8510,0.8240,6m 36.4s,6.6068,396.4103
LightGBM_base,LightGBM_base,NaN,NaN,0.9982,0.9161,NaN,0.5982,0.7772,3.8s,0.0636,3.8144
XGBoost_sin_smote,XGBoost_sin_smote,XGBoost,False,0.9981,0.9121,0.9156,0.5719,0.7601,59.2s,0.9861,59.1688
XGBoost_con_smote,XGBoost_con_smote,XGBoost,True,0.9979,0.9109,0.9140,0.5121,0.7161,14m 13.4s,14.2231,853.3863
XGBoost_base,XGBoost_base,NaN,NaN,0.9979,0.9010,NaN,0.4936,0.7031,1.8s,0.0308,1.8490
RandomForest_sin_smote,RandomForest_sin_smote,RandomForest,False,0.9972,0.8966,0.8993,0.6788,0.8100,22m 06.4s,22.1072,1326.4340
RandomForest_con_smote,RandomForest_con_smote,RandomForest,True,0.9974,0.8931,0.8928,0.6725,0.8067,1h 57m 54.3s,117.9051,7074.3031
RandomForest_base,RandomForest_base,NaN,NaN,0.9972,0.8499,NaN,0.4475,0.6652,6.0s,0.0998,5.9877


Comparativa resumida con/sin SMOTE:


pr_auc      f2  fit_time_min
algorithm    smote                              
LightGBM     False  0.9266  0.8240        6.6068
             True   0.9269  0.8433       41.5667
RandomForest False  0.8966  0.8100       22.1072
             True   0.8931  0.8067      117.9051
XGBoost      False  0.9121  0.7601        0.9861
             True   0.9109  0.7161       14.2231

Mejor modelo: LightGBM_con_smote
PR-AUC test: 0.9269
ROC-AUC test: 0.9984
F2 test: 0.8433
Tiempo entrenamiento: 41m 34.0s

Classification report:
              precision    recall  f1-score   support

    legitima       1.00      1.00      1.00    117489
      fraude       0.87      0.84      0.85      1440

    accuracy                           1.00    118929
   macro avg       0.94      0.92      0.93    118929
weighted avg       1.00      1.00      1.00    118929

Matriz de confusión:
[[117314    175]
 [   236   1204]]
Resultados guardados en: /Users/aldan/Desktop/tfg_code/models/model_comparison.csv
Modelo guardado en: /Users/aldan/Desktop/tfg_code/models/best_fraud_pipeline.joblib


## Interpretación de resultados supervisados

El mejor modelo obtenido es `LightGBM_con_smote`, con **PR-AUC = 0.9269**, **ROC-AUC = 0.9984**, **F2 = 0.8433** y un tiempo de entrenamiento de **41m 34.0s**. En test detecta **1204 de 1440 fraudes**, dejando **236 fraudes sin detectar** y generando **175 falsos positivos**.

La métrica más importante para comparar estos modelos es **PR-AUC**, porque la clase fraude representa solo un **1.211%** del dataset. En este contexto, ROC-AUC es útil como referencia, pero puede resultar demasiado optimista: todos los modelos están cerca de 0.997-0.998 aunque existen diferencias relevantes en detección de fraude.

Mi lectura es que **LightGBM es claramente la mejor familia de modelos para este problema**. La diferencia entre `LightGBM_con_smote` y `LightGBM_sin_smote` es muy pequeña en PR-AUC (**0.9269 vs 0.9266**), aunque SMOTENC mejora el F2 (**0.8433 vs 0.8240**), lo que indica que ayuda algo a recuperar más fraude. El coste es importante: pasa de **6m 36.4s** sin SMOTE a **41m 34.0s** con SMOTE. Por tanto, si el objetivo es máxima detección, escogería `LightGBM_con_smote`; si se prioriza eficiencia y simplicidad, `LightGBM_sin_smote` es casi igual de bueno.

Random Forest mejora con Grid Search, pero queda por debajo de LightGBM y tarda bastante, especialmente con SMOTE. XGBoost también es competitivo, pero no supera a LightGBM y SMOTE no le ayuda en este experimento. En resumen: **SMOTENC solo parece útil para LightGBM**, y aun así la mejora en PR-AUC es marginal frente al aumento de tiempo.

## Stacking de modelos supervisados

A continuación se construye un modelo de stacking con los mejores modelos ajustados. La idea es que un metamodelo de regresión logística aprenda a combinar las probabilidades de los clasificadores base. No está garantizado que supere a LightGBM, porque el mejor modelo individual ya es muy fuerte, pero sirve para comprobar si la combinación de modelos captura señales complementarias.

In [67]:
from sklearn.base import clone
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

STACKING_TOP_N = 3
STACKING_CV_FOLDS = 3

stacking_candidates = comparison_df.dropna(subset=["algorithm"]).head(STACKING_TOP_N).index.tolist()
print("Modelos usados en stacking:", stacking_candidates)

stacking_estimators = [
    (name.lower().replace(" ", "_"), clone(all_estimators[name]))
    for name in stacking_candidates
]

stacking_model = StackingClassifier(
    estimators=stacking_estimators,
    final_estimator=LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE,
    ),
    stack_method="predict_proba",
    cv=StratifiedKFold(
        n_splits=STACKING_CV_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE,
    ),
    n_jobs=N_JOBS,
    passthrough=False,
)

start = time.perf_counter()
stacking_model.fit(X_train, y_train)
stacking_time = time.perf_counter() - start

stacking_metrics = evaluate_model("Stacking_top3", stacking_model, X_test, y_test)
stacking_metrics["algorithm"] = "Stacking"
stacking_metrics["smote"] = "mixto"
stacking_metrics["fit_time_sec"] = stacking_time
stacking_metrics["best_cv_pr_auc"] = np.nan
stacking_metrics["best_params"] = {"base_models": stacking_candidates}

all_results["Stacking_top3"] = stacking_metrics
all_estimators["Stacking_top3"] = stacking_model

comparison_df = summarize_results(all_results)
display(
    comparison_df[
        [
            col
            for col in [
                "model",
                "algorithm",
                "smote",
                "roc_auc",
                "pr_auc",
                "best_cv_pr_auc",
                "f1",
                "f2",
                "fit_time",
                "fit_time_min",
            ]
            if col in comparison_df.columns
        ]
    ]
)

print("Stacking PR-AUC:", round(stacking_metrics["pr_auc"], 4))
print("Stacking ROC-AUC:", round(stacking_metrics["roc_auc"], 4))
print("Stacking F2:", round(stacking_metrics["f2"], 4))
print("Tiempo stacking:", format_seconds(stacking_time))
print("\nClassification report stacking:")
print(stacking_metrics["classification_report"])
print("Matriz de confusión stacking:")
print(stacking_metrics["confusion_matrix"])

best_name = comparison_df.index[0]
best_pipeline = all_estimators[best_name]
comparison_output = comparison_df.drop(columns=["confusion_matrix", "classification_report"], errors="ignore")
comparison_output.to_csv(models_path / "model_comparison_with_stacking.csv", index=True)
joblib.dump(stacking_model, models_path / "stacking_fraud_pipeline.joblib")
joblib.dump(best_pipeline, models_path / "best_fraud_pipeline.joblib")

print("Mejor modelo tras stacking:", best_name)
print("Resultados con stacking guardados en:", models_path / "model_comparison_with_stacking.csv")
print("Stacking guardado en:", models_path / "stacking_fraud_pipeline.joblib")

Modelos usados en stacking: ['LightGBM_con_smote', 'LightGBM_sin_smote', 'XGBoost_sin_smote']


,model,algorithm,smote,roc_auc,pr_auc,best_cv_pr_auc,f1,f2,fit_time,fit_time_min
LightGBM_con_smote,LightGBM_con_smote,LightGBM,True,0.9984,0.9269,0.9275,0.8542,0.8433,41m 34.0s,41.5667
LightGBM_sin_smote,LightGBM_sin_smote,LightGBM,False,0.9985,0.9266,0.9261,0.8510,0.8240,6m 36.4s,6.6068
Stacking_top3,Stacking_top3,Stacking,mixto,0.9982,0.9233,NaN,0.5687,0.7586,1m 04.1s,1.0682
LightGBM_base,LightGBM_base,NaN,NaN,0.9982,0.9161,NaN,0.5982,0.7772,3.8s,0.0636
XGBoost_sin_smote,XGBoost_sin_smote,XGBoost,False,0.9981,0.9121,0.9156,0.5719,0.7601,59.2s,0.9861
XGBoost_con_smote,XGBoost_con_smote,XGBoost,True,0.9979,0.9109,0.9140,0.5121,0.7161,14m 13.4s,14.2231
XGBoost_base,XGBoost_base,NaN,NaN,0.9979,0.9010,NaN,0.4936,0.7031,1.8s,0.0308
RandomForest_sin_smote,RandomForest_sin_smote,RandomForest,False,0.9972,0.8966,0.8993,0.6788,0.8100,22m 06.4s,22.1072
RandomForest_con_smote,RandomForest_con_smote,RandomForest,True,0.9974,0.8931,0.8928,0.6725,0.8067,1h 57m 54.3s,117.9051
RandomForest_base,RandomForest_base,NaN,NaN,0.9972,0.8499,NaN,0.4475,0.6652,6.0s,0.0998


Stacking PR-AUC: 0.9233
Stacking ROC-AUC: 0.9982
Stacking F2: 0.7586
Tiempo stacking: 1m 04.1s

Classification report stacking:
              precision    recall  f1-score   support

    legitima       1.00      0.98      0.99    117489
      fraude       0.40      0.98      0.57      1440

    accuracy                           0.98    118929
   macro avg       0.70      0.98      0.78    118929
weighted avg       0.99      0.98      0.99    118929

Matriz de confusión stacking:
[[115393   2096]
 [    35   1405]]
Mejor modelo tras stacking: LightGBM_con_smote
Resultados con stacking guardados en: /Users/aldan/Desktop/tfg_code/models/model_comparison_with_stacking.csv
Stacking guardado en: /Users/aldan/Desktop/tfg_code/models/stacking_fraud_pipeline.joblib


## Interpretación esperada del stacking

Para considerar que el stacking mejora realmente, debería superar a `LightGBM_con_smote` en **PR-AUC** y, preferiblemente, también en **F2**. Si el stacking queda igual o peor, la conclusión sigue siendo útil: indica que LightGBM ya captura la mayor parte de la señal predictiva y que combinar modelos añade complejidad sin beneficio suficiente.

En problemas de fraude muy desbalanceados, no conviene elegir el stacking solo porque sea más sofisticado. Si la mejora es mínima, se debe priorizar el modelo individual más simple, rápido e interpretable operacionalmente.

## Revisión de los mejores parámetros

In [68]:
for model_name, result in grid_results.items():
    print("=" * 80)
    print(model_name)
    print("=" * 80)
    print(result["best_params"])
    display(
        cv_results[model_name][
            [
                "rank_test_pr_auc",
                "mean_test_pr_auc",
                "mean_test_roc_auc",
                "mean_test_f2",
                "mean_fit_time",
                "params",
            ]
        ].head(10)
    )

RandomForest_sin_smote
{'classifier__class_weight': 'balanced_subsample', 'classifier__max_depth': 25, 'classifier__max_features': 0.5, 'classifier__min_samples_leaf': 5, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 400}


,rank_test_pr_auc,mean_test_pr_auc,mean_test_roc_auc,mean_test_f2,mean_fit_time,params
31,1,0.899308,0.997544,0.820065,172.829345,{'classifier__class_weight': 'balanced_subsamp...
29,1,0.899308,0.997544,0.820065,170.897863,{'classifier__class_weight': 'balanced_subsamp...
33,3,0.899102,0.995605,0.793108,153.209236,{'classifier__class_weight': 'balanced_subsamp...
30,4,0.898551,0.997038,0.820527,85.578269,{'classifier__class_weight': 'balanced_subsamp...
28,4,0.898551,0.997038,0.820527,84.926206,{'classifier__class_weight': 'balanced_subsamp...
47,6,0.898185,0.997266,0.832352,109.868086,{'classifier__class_weight': 'balanced_subsamp...
45,6,0.898185,0.997266,0.832352,188.230725,{'classifier__class_weight': 'balanced_subsamp...
44,8,0.897469,0.996506,0.832460,96.426551,{'classifier__class_weight': 'balanced_subsamp...
46,8,0.897469,0.996506,0.832460,94.927906,{'classifier__class_weight': 'balanced_subsamp...
32,10,0.896676,0.994599,0.791658,83.506704,{'classifier__class_weight': 'balanced_subsamp...


RandomForest_con_smote
{'classifier__class_weight': 'balanced_subsample', 'classifier__max_depth': 25, 'classifier__max_features': 0.5, 'classifier__min_samples_leaf': 5, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 400, 'smote__k_neighbors': 5, 'smote__sampling_strategy': 0.05}


,rank_test_pr_auc,mean_test_pr_auc,mean_test_roc_auc,mean_test_f2,mean_fit_time,params
118,1,0.892849,0.997489,0.814579,213.983086,{'classifier__class_weight': 'balanced_subsamp...
126,1,0.892849,0.997489,0.814579,216.706487,{'classifier__class_weight': 'balanced_subsamp...
122,3,0.892480,0.997321,0.814205,112.383363,{'classifier__class_weight': 'balanced_subsamp...
114,3,0.892480,0.997321,0.814205,112.166179,{'classifier__class_weight': 'balanced_subsamp...
124,5,0.892393,0.997228,0.814357,216.049071,{'classifier__class_weight': 'balanced_subsamp...
116,5,0.892393,0.997228,0.814357,216.594971,{'classifier__class_weight': 'balanced_subsamp...
190,7,0.891996,0.997166,0.824154,233.269121,{'classifier__class_weight': 'balanced_subsamp...
182,7,0.891996,0.997166,0.824154,246.525539,{'classifier__class_weight': 'balanced_subsamp...
112,9,0.891932,0.997137,0.813163,111.337530,{'classifier__class_weight': 'balanced_subsamp...
120,9,0.891932,0.997137,0.813163,107.750976,{'classifier__class_weight': 'balanced_subsamp...


LightGBM_sin_smote
{'classifier__class_weight': None, 'classifier__colsample_bytree': 0.8, 'classifier__learning_rate': 0.03, 'classifier__max_depth': -1, 'classifier__min_child_samples': 20, 'classifier__n_estimators': 600, 'classifier__num_leaves': 31, 'classifier__subsample': 0.8}


...

,rank_test_pr_auc,mean_test_pr_auc,mean_test_roc_auc,mean_test_f2,mean_fit_time,params
154,1,0.927461,0.998452,0.839360,23.026778,"{'classifier__class_weight': None, 'classifier..."
210,2,0.926799,0.998439,0.834827,18.824790,"{'classifier__class_weight': None, 'classifier..."
208,3,0.926325,0.998435,0.836670,12.300272,"{'classifier__class_weight': None, 'classifier..."
152,4,0.926144,0.998434,0.838010,18.044961,"{'classifier__class_weight': None, 'classifier..."
218,5,0.926032,0.998435,0.833126,22.578600,"{'classifier__class_weight': None, 'classifier..."
136,6,0.925902,0.998403,0.836408,25.344416,"{'classifier__class_weight': None, 'classifier..."
158,7,0.925824,0.998429,0.831075,29.741151,"{'classifier__class_weight': None, 'classifier..."
216,8,0.925743,0.998420,0.834300,18.213940,"{'classifier__class_weight': None, 'classifier..."
138,9,0.925640,0.998402,0.836031,23.857253,"{'classifier__class_weight': None, 'classifier..."
140,10,0.925613,0.998423,0.831403,22.927294,"{'classifier__class_weight': None, 'classifier..."


XGBoost_sin_smote
{'classifier__colsample_bytree': 0.8, 'classifier__learning_rate': 0.07, 'classifier__max_depth': 5, 'classifier__min_child_weight': 5, 'classifier__n_estimators': 600, 'classifier__scale_pos_weight': np.float64(81.5892361111111), 'classifier__subsample': 0.8}


,rank_test_pr_auc,mean_test_pr_auc,mean_test_roc_auc,mean_test_f2,mean_fit_time,params
30,1,0.915650,0.998243,0.780759,5.829466,"{'classifier__colsample_bytree': 0.8, 'classif..."
26,2,0.914478,0.998193,0.782760,6.660318,"{'classifier__colsample_bytree': 0.8, 'classif..."
31,3,0.913840,0.998210,0.753435,4.657802,"{'classifier__colsample_bytree': 0.8, 'classif..."
27,4,0.912330,0.998161,0.755418,5.630890,"{'classifier__colsample_bytree': 0.8, 'classif..."
22,5,0.909561,0.998137,0.735489,5.576020,"{'classifier__colsample_bytree': 0.8, 'classif..."
28,6,0.909382,0.998105,0.732256,3.951024,"{'classifier__colsample_bytree': 0.8, 'classif..."
18,7,0.909289,0.998120,0.734775,5.992312,"{'classifier__colsample_bytree': 0.8, 'classif..."
24,8,0.908462,0.998072,0.736815,3.907130,"{'classifier__colsample_bytree': 0.8, 'classif..."
29,9,0.908096,0.998074,0.714974,3.293305,"{'classifier__colsample_bytree': 0.8, 'classif..."
14,10,0.907828,0.998061,0.723135,6.821450,"{'classifier__colsample_bytree': 0.8, 'classif..."


XGBoost_con_smote
{'classifier__colsample_bytree': 0.8, 'classifier__learning_rate': 0.07, 'classifier__max_depth': 5, 'classifier__min_child_weight': 5, 'classifier__n_estimators': 600, 'classifier__scale_pos_weight': np.float64(81.5892361111111), 'classifier__subsample': 0.8, 'smote__k_neighbors': 3, 'smote__sampling_strategy': 0.05}


,rank_test_pr_auc,mean_test_pr_auc,mean_test_roc_auc,mean_test_f2,mean_fit_time,params
120,1,0.913961,0.998146,0.750033,17.391591,"{'classifier__colsample_bytree': 0.8, 'classif..."
122,2,0.913578,0.998137,0.748990,20.882321,"{'classifier__colsample_bytree': 0.8, 'classif..."
126,3,0.913443,0.998151,0.725272,13.590969,"{'classifier__colsample_bytree': 0.8, 'classif..."
124,4,0.913181,0.998152,0.722718,15.896815,"{'classifier__colsample_bytree': 0.8, 'classif..."
106,5,0.912421,0.998101,0.752354,22.496455,"{'classifier__colsample_bytree': 0.8, 'classif..."
104,6,0.912314,0.998083,0.751250,19.016609,"{'classifier__colsample_bytree': 0.8, 'classif..."
121,7,0.912144,0.998109,0.724197,28.337959,"{'classifier__colsample_bytree': 0.8, 'classif..."
108,8,0.912118,0.998089,0.726496,16.673241,"{'classifier__colsample_bytree': 0.8, 'classif..."
110,9,0.911806,0.998112,0.728080,26.233656,"{'classifier__colsample_bytree': 0.8, 'classif..."
125,10,0.911050,0.998078,0.703502,23.422786,"{'classifier__colsample_bytree': 0.8, 'classif..."
